# Notebook 01 — Data Exploration & Preprocessing

This notebook walks through:
- Loading a 3D medical volume (NIfTI / DICOM)
- Inspecting spatial metadata (spacing, origin, direction)
- Visualizing axial, sagittal, and coronal slices
- Resampling to isotropic spacing
- Intensity normalization
- Comparing fixed vs. moving volume before registration

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk

from src.io.loaders import load_volume, sitk_to_numpy
from src.preprocessing.preprocess import resample_to_spacing, normalize_intensity

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load volumes

Update the paths below to point to your downloaded dataset.
See `data/README.md` for download instructions.

In [ ]:
FIXED_PATH  = '../data/raw/fixed.nii.gz'
MOVING_PATH = '../data/raw/moving.nii.gz'

fixed  = load_volume(FIXED_PATH)
moving = load_volume(MOVING_PATH)

## 2. Inspect spatial metadata

In [ ]:
for name, img in [('Fixed', fixed), ('Moving', moving)]:
    print(f"--- {name} ---")
    print(f"  Size (x,y,z):  {img.GetSize()}")
    print(f"  Spacing (mm):  {img.GetSpacing()}")
    print(f"  Origin  (mm):  {img.GetOrigin()}")
    print(f"  Direction:     {img.GetDirection()}")
    arr = sitk_to_numpy(img)
    print(f"  Intensity:     [{arr.min():.1f}, {arr.max():.1f}]  mean={arr.mean():.1f}")
    print()

## 3. Multi-plane visualization

In [ ]:
def show_multiplane(image, title='Volume', cmap='gray'):
    arr = sitk_to_numpy(image)  # (z, y, x)
    z, y, x = arr.shape
    vmin, vmax = np.percentile(arr, [1, 99])

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    axes[0].imshow(arr[z//2, :, :], cmap=cmap, vmin=vmin, vmax=vmax, origin='lower')
    axes[0].set_title(f'Axial (z={z//2})')
    axes[1].imshow(arr[:, y//2, :], cmap=cmap, vmin=vmin, vmax=vmax, origin='lower')
    axes[1].set_title(f'Coronal (y={y//2})')
    axes[2].imshow(arr[:, :, x//2], cmap=cmap, vmin=vmin, vmax=vmax, origin='lower')
    axes[2].set_title(f'Sagittal (x={x//2})')
    for ax in axes:
        ax.axis('off')
    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_multiplane(fixed, 'Fixed volume')
show_multiplane(moving, 'Moving volume')

## 4. Resampling to isotropic spacing

In [ ]:
TARGET_SPACING = (1.0, 1.0, 1.0)  # mm

fixed_resampled  = resample_to_spacing(fixed, TARGET_SPACING)
moving_resampled = resample_to_spacing(moving, TARGET_SPACING)

print(f"Fixed  before: {fixed.GetSize()} @ {fixed.GetSpacing()}")
print(f"Fixed  after:  {fixed_resampled.GetSize()} @ {fixed_resampled.GetSpacing()}")
print()
print(f"Moving before: {moving.GetSize()} @ {moving.GetSpacing()}")
print(f"Moving after:  {moving_resampled.GetSize()} @ {moving_resampled.GetSpacing()}")

## 5. Intensity normalization

In [ ]:
fixed_norm  = normalize_intensity(fixed_resampled, method='zscore')
moving_norm = normalize_intensity(moving_resampled, method='zscore')

# Show histograms before / after
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (img_before, img_after, name) in zip(
    axes,
    [(fixed_resampled, fixed_norm, 'Fixed'), (moving_resampled, moving_norm, 'Moving')]
):
    arr_before = sitk_to_numpy(img_before).flatten()
    arr_after  = sitk_to_numpy(img_after).flatten()
    ax.hist(arr_before, bins=100, alpha=0.5, label='Original', color='steelblue')
    ax.hist(arr_after,  bins=100, alpha=0.5, label='Normalized', color='salmon')
    ax.legend()
    ax.set_title(f'{name} — Intensity histogram')
    ax.set_xlabel('Intensity')
    ax.set_ylabel('Voxel count')
plt.tight_layout()
plt.show()

## 6. Side-by-side: Fixed vs Moving (before registration)

Visual inspection of misalignment before registration.

In [ ]:
arr_f = sitk_to_numpy(fixed_norm)
arr_m = sitk_to_numpy(moving_norm)

z = arr_f.shape[0] // 2
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(arr_f[z], cmap='gray', origin='lower')
axes[0].set_title(f'Fixed — z={z}', fontweight='bold')
axes[1].imshow(arr_m[z], cmap='gray', origin='lower')
axes[1].set_title(f'Moving (before reg.) — z={z}', fontweight='bold')
for ax in axes:
    ax.axis('off')
plt.suptitle('Initial misalignment', fontsize=12)
plt.tight_layout()
plt.show()